<a href="https://colab.research.google.com/github/zhaoyingjun/Tiny-R2/blob/main/Tiny_R2_journey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tiktoken datasets transformers huggingface_hub
!pip install git+https://github.com/KellerJordan/Muon
!pip install --upgrade transformers


  Cloning https://github.com/KellerJordan/Muon to /tmp/pip-req-build-jkxscljc
  Running command git clone --filter=blob:none --quiet https://github.com/KellerJordan/Muon /tmp/pip-req-build-jkxscljc
  Resolved https://github.com/KellerJordan/Muon to commit 6399c658d3c4a3356ba823fa6664b10e23871068
  Preparing metadata (setup.py) ... done
  Created wheel for muon-optimizer: filename=muon_optimizer-0.1.0-py3-none-any.whl size=7141 sha256=59c5ef94fd7699b5ad026fc9a6b31835e3e631de2756e69cf28c43d27df4de4b
  Stored in directory: /tmp/pip-ephem-wheel-cache-4wwbpgt4/wheels/6e/33/94/64d18603ba0f39064aab523d6edf493c388cfb7419bb5c9043
Successfully built muon-optimizer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 150.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install faiss-cpu sentence_transformers datasets rouge-score nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 155.9 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=08e8806a2b6d3440f3cedb49a52cdfa5c9a82ce2a51eedfa5dd45b28f7682578
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
!hf auth login --force

In [3]:
!git clone https://github.com/zhaoyingjun/Tiny-R2.git
%cd Tiny-R2

Cloning into 'Tiny-R2'...
remote: Enumerating objects: 436, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 436 (delta 62), reused 0 (delta 0), pack-reused 321 (from 2)
Receiving objects: 100% (436/436), 3.96 MiB | 1.33 MiB/s, done.
Resolving deltas: 100% (243/243), done.
/content/Tiny-R2


**训练climbmix数据，可以训练一个基础大模型，1.7B参数量，评测模型结构和效果。支持采用大模型作为Agent进行观察训练过程并给出实施的lr、clip的调整参数;--use_agent_observe=True开启,默认是关闭的，需要输入gemini的api_key**

默认不开启Agent智能化自主训练

In [ ]:
!python train.py --n_layer 6 --n_embd 1536 --hc 'True' --mhc 'True' --n_experts 8 --max_iters 10000 --attention_types 'Sparse' --batch_size 8 --ctx_len 2048 --hf_dataset 'karpathy/climbmix-400b-shuffle' --resume True --save_best_only True

开启Agent智能化自主训练，需要将your gemini api key替换

In [ ]:
!python train.py --n_layer 6 --n_embd 1536 --hc 'True' --mhc 'True' --n_experts 8 --max_iters 10000 --attention_types 'Sparse' --batch_size 8 --ctx_len 2048 --hf_dataset 'karpathy/climbmix-400b-shuffle' --resume True --save_best_only True --use_agent_observe True --gemini_api_key 'your gemini api key'

预训练完成之后，进行benchmark的验证

In [ ]:
!python /content/Tiny-R2/evaluate.py --checkpoint /content/Tiny-R2/checkpoints/best_model_step_1180.pt

后训练直接使用Qwen3.5-9B模型作为教师模型进行OPD训练，可自定义使用RAG增加教师模型以及自定义数据集等

In [ ]:
!python opd_train.py --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name tiny-r2 \
  --student_ckpt checkpoints/best_model_step_40.pt \
  --enable_rag_teacher \
  --batch_size 2 \
  --grad_accum_steps 4 \
  --custom_qa_path baoxianqa.jsonl \
  --rag_corpus_path baoxianqa.txt

✅ 成功导入 Tiny-R2 核心模块 (将作为备用学生模型)
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
💡 [System] 已自动初始化单进程 Gloo 分布式环境，以兼容 Muon 优化器。

🚀 正在启动 Agent OPD 蒸馏训练
👨‍🏫 Teacher 模型: Qwen/Qwen3.5-9B (外挂 RAG)
👶 Student 模型: tiny-r2 (无 RAG, 在线 Rollout 比率=0.7)

🔧 已将 Tiny-R2 词表大小 (vocab_size) 动态同步为: 248077
📂 正在加载本地自定义数据集: baoxianqa.jsonl
Generating train split: 42 examples [00:00, 14484.52 examples/s]
📥 正在从数据集中提取知识，充当 RAG 背景语料...
🔍 正在初始化 RAG 检索器 (shibing624/text2vec-base-chinese)...
modules.json: 100% 230/230 [00:00<00:00, 2.66MB/s]
README.md: 13.7kB [00:00, 71.6MB/s]
sentence_bert_config.json: 100% 54.0/54.0 [00:00<00:00, 708kB/s]
config.json: 100% 856/856 [00:00<00:00, 12.0MB/s]
model.safetensors: 100% 409M/409M [00:04<00:00, 102MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 15426.79it/s]
tokenizer_config.json: 100% 319/319 [00:00<00:00, 3.31MB/s]
vocab.txt: 110kB [00:00, 14.7MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 1.51MB/s]
config.json: 100% 

In [ ]:
!pip install flash-linear-attention causal-conv1d

使用Qwen3.5-9B模型作为教师模型、Qwen3.5-0.8B作为学生模型进行OPD训练，用RAG增加教师模型，RAG数据集来自问答数据集集medquad，可以复现Readme中的结果;去除命令行中--enable_reg_teacher则关闭外挂RAG功能；--rag_corpus_path外挂RAG数据集，--custom_qa_path 自定义问题集

In [12]:
!python opd_train.py \
  --dataset medquad \
  --enable_rag_teacher\
  --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name Qwen/Qwen3.5-0.8B \
  --batch_size 2 \
  --grad_accum_steps 4


✅ 成功导入 Tiny-R2 核心模块 (将作为备用学生模型)
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
💡 [System] 已自动初始化单进程 Gloo 分布式环境，以兼容 Muon 优化器。

🚀 正在启动 Agent OPD 蒸馏训练
👨‍🏫 Teacher 模型: Qwen/Qwen3.5-9B (外挂 RAG)
👶 Student 模型: Qwen/Qwen3.5-0.8B (无 RAG, 在线 Rollout 比率=0.7)

🔧 已将 Tiny-R2 词表大小 (vocab_size) 动态同步为: 248077
📡 正在从 HuggingFace 加载公共数据集: medquad
📥 正在从数据集中提取知识，充当 RAG 背景语料...
🔍 正在初始化 RAG 检索器 (sentence-transformers/all-MiniLM-L6-v2)...
Loading weights: 100% 103/103 [00:00<00:00, 15736.47it/s]
Batches: 100% 32/32 [00:00<00:00, 71.90it/s]
✅ RAG 向量库构建完成，共包含 1000 条医学知识。

⚡ 检测到 GPU 支持 bfloat16，将使用 bfloat16 混合精度 (自动禁用 GradScaler)

🤖 加载 HuggingFace 模型作为学生: Qwen/Qwen3.5-0.8B
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github

默认使用medquade数据集进行OPD的结果验证，可以根据OPD训练的数据集对应的修改验证

In [ ]:
!python eval_opd.py \
    --dataset medquad \
    --hf_teacher_model Qwen/Qwen3.5-9B \
    --student_base_model Qwen/Qwen3.5-0.8B \
    --student_opd_model opd_checkpoints/student_best_model_step_80.pt \
    --eval_samples 10